# Урок 2. Экосистема Hugging Face

🎯 **Цели урока**

- сравнить токенизаторы и посчитать, во что обходится русский текст;
- применить готовую модель через `pipeline` — и поймать её на контаминации;
- получить контекстные эмбеддинги через `AutoTokenizer` + `AutoModel`;
- загрузить и подготовить данные библиотекой `datasets`;
- проверить модель на Hub: лицензия, данные обучения, фиксированная ревизия.

📖 **Теория:** разделы «Токенизация», «Экосистема Hugging Face», «Лицензии моделей и данных».

⏱ Первый запуск скачивает ~250 МБ: две маленькие модели BERT и токенизаторы GPT-2 и Qwen3.
Модели лежат в кэше `~/.cache/huggingface` и повторно не качаются.

In [1]:
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")  # tqdm в Jupyter без ipywidgets

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import torch  # noqa: E402
import torch.nn.functional as F  # noqa: E402
from datasets import disable_progress_bars, load_dataset  # noqa: E402
from huggingface_hub import model_info  # noqa: E402
from huggingface_hub.utils import logging as hub_logging  # noqa: E402
from sklearn.metrics import accuracy_score  # noqa: E402
from transformers import AutoModel, AutoTokenizer, pipeline  # noqa: E402
from transformers.utils import logging as hf_logging  # noqa: E402

from mlcourse import fetch, get_device  # noqa: E402

hf_logging.set_verbosity_error()
hub_logging.set_verbosity_error()  # без напоминаний про HF_TOKEN: публичные модели качаются и так
disable_progress_bars()  # прогресс-бары datasets засоряют вывод ноутбука
device = get_device()
folder = fetch("rureviews")
test = pd.read_json(folder / "test.jsonl", lines=True)
print("устройство:", device)

устройство: mps


## 1. Токенизаторы

Возьмём три токенизатора: GPT-2 (обучен на английском), Qwen3 (многоязычный, как модели курса)
и rubert-tiny2 (обучен на русском). `AutoTokenizer` скачивает только файлы токенизатора, не веса.

In [2]:
tokenizers = {
    "GPT-2": AutoTokenizer.from_pretrained("openai-community/gpt2"),
    "Qwen3": AutoTokenizer.from_pretrained("Qwen/Qwen3-8B"),
    "rubert-tiny2": AutoTokenizer.from_pretrained("cointegrated/rubert-tiny2"),
}
text = "Платье маломерит, пришлось вернуть."
for name, tok in tokenizers.items():
    ids = tok(text, add_special_tokens=False)["input_ids"]
    pieces = [tok.decode([i]) for i in ids]
    print(f"{name:12} словарь {len(tok):>7,} | {len(ids):2} токенов: {pieces}".replace(",", " "))

GPT-2        словарь  50 257 | 37 токенов: ['�'  '�'  'л'  'а'  'т'  'ь'  'е'  ' �'  '�'  'а'  'л'  'о�'  '�'  'е'  'р'  'и'  'т'  ' '  ' �'  '�'  'р'  'и'  '�'  '�'  'л'  'о'  'с'  'ь'  ' �'  '�'  'е'  'р'  'н'  'у'  'т'  'ь'  '.']
Qwen3        словарь 151 669 | 14 токенов: ['П'  'л'  'ать'  'е'  ' мал'  'омер'  'ит'  ' '  ' при'  'ш'  'лось'  ' вер'  'нуть'  '.']
rubert-tiny2 словарь  83 828 |  9 токенов: ['Плат'  '##ье'  'малом'  '##ери'  '##т'  ' '  'пришлось'  'вернуть'  '.']


GPT-2 режет кириллицу на байты: одна буква — два байта UTF-8, и многие куски по отдельности даже
не печатаются (`�`). Qwen3 знает частые русские слова и их куски. rubert-tiny2 знает русский лучше всех,
но зато хуже справится с английским и кодом.

Сколько это стоит на реальных данных? Посчитаем токены на тысяче отзывов.

In [3]:
reviews = test.text.sample(1000, random_state=0).tolist()
rows = []
for name, tok in tokenizers.items():
    n_tokens = sum(len(tok(r, add_special_tokens=False)["input_ids"]) for r in reviews)
    n_words = sum(len(r.split()) for r in reviews)
    rows.append({"токенизатор": name, "токенов на отзыв": n_tokens / len(reviews), "токенов на слово": n_tokens / n_words})
pd.DataFrame(rows).set_index("токенизатор").round(2)

,токенов на отзыв,токенов на слово
токенизатор,,
GPT-2,153.07,6.86
Qwen3,52.98,2.37
rubert-tiny2,34.24,1.53


Для GPT-2 русский отзыв в несколько раз «дороже», чем для Qwen3: больше денег за API, медленнее
генерация, меньше текста помещается в контекст. Считать токены на своих данных — обязательный шаг
при выборе модели.

### Специальные токены и чат-шаблон

Токенизатор BERT обрамляет текст служебными токенами. У чат-модели в токенизаторе хранится
**чат-шаблон** — то, во что превращается список сообщений перед подачей в модель.

In [4]:
bert_tok = tokenizers["rubert-tiny2"]
print(bert_tok.convert_ids_to_tokens(bert_tok("Отличное платье!")["input_ids"]))

messages = [{"role": "system", "content": "Ты — вежливый помощник."}, {"role": "user", "content": "Привет!"}]
print(tokenizers["Qwen3"].apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                              enable_thinking=False))

['[CLS]', 'Отлично', '##е', 'платье', '!', '[SEP]']
<|im_start|>system
Ты — вежливый помощник.<|im_end|>
<|im_start|>user
Привет!<|im_end|>
<|im_start|>assistant
<think>

</think>




Вот что на самом деле получает модель от OpenAI-совместимого API из M00: сообщения, склеенные
в одну строку со специальными токенами `<|im_start|>` и `<|im_end|>`. Пустой блок `<think></think>` —
это и есть выключенные рассуждения (`no_thinking()`). Подробно — в M06 и M09.

### Батч: паддинг и маска внимания

In [5]:
batch = bert_tok(["Супер!", "Платье пришло быстро, но маломерит."], padding=True, return_tensors="pt")
print(batch["input_ids"])
print(batch["attention_mask"])

tensor([[    2, 35565,     5,     3,     0,     0,     0,     0,     0,     0,
             0,     0],
        [    2, 35176,  5976, 34941, 15398,    16,  1363, 67392,  6854,   751,
            18,     3]])
tensor([[1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


Короткий текст дополнен нулями (`[PAD]`), а `attention_mask` говорит модели не смотреть на эти позиции.

## 2. pipeline — и первая ловушка

`pipeline` — самый короткий путь от текста к ответу. Возьмём готовый классификатор тональности
на основе rubert-tiny2 (лицензия MIT).

In [6]:
classifier = pipeline("text-classification", model="seara/rubert-tiny2-russian-sentiment", device=device)
print(classifier(["Прекрасное платье, спасибо!", "Ткань так себе, но за эти деньги нормально.", "Не пришло."]))
print("метки модели:", classifier.model.config.id2label)

[{'label': 'positive', 'score': 0.9959523677825928}, {'label': 'neutral', 'score': 0.6127701997756958}, {'label': 'negative', 'score': 0.7797757387161255}]
метки модели: {0: 'neutral', 1: 'positive', 2: 'negative'}


Обратите внимание на порядок меток: у модели `0 = neutral`, а в нашем датасете `0 = negative`.
Сравнивать нужно по названиям, а не по номерам. Проверим модель на тесте RuReviews:

In [7]:
sample = test.sample(3000, random_state=0)
preds = [p["label"] for p in classifier(sample.text.tolist(), batch_size=64, truncation=True)]
print(f"готовая модель на тесте RuReviews: accuracy {accuracy_score(sample.label_text, preds):.3f}")
print("наш лучший baseline из урока 1:   accuracy 0.752")

готовая модель на тесте RuReviews: accuracy 0.792
наш лучший baseline из урока 1:   accuracy 0.752


Готовая маленькая модель обгоняет наш baseline на несколько пунктов. Хорошо? Прежде чем радоваться,
посмотрим, на чём она обучалась. Карточка модели доступна и через API.

In [8]:
info = model_info("seara/rubert-tiny2-russian-sentiment")
print("лицензия:", info.card_data.license)
print("данные обучения:", info.card_data.datasets)

лицензия: mit
данные обучения: ['sismetanin/rureviews', 'RuSentiment', 'LinisCrowd2015', 'LinisCrowd2016', 'KaggleRussianNews']


Среди данных обучения — `sismetanin/rureviews`, то есть наш датасет. Модель обучалась на объединении
нескольких датасетов со своим разбиением на train и test, и отзывы из нашего теста почти наверняка
были у неё в обучении. Это **контаминация** (M02): оценка на таком тесте завышена, и сравнивать
модель с нашим baseline нечестно. Правило: прежде чем доверять цифрам готовой модели на своём тесте,
прочитайте, на чём она обучалась.

## 3. AutoModel: контекстные эмбеддинги

Уровнем ниже — токенизатор и «голое» тело модели. Выход — вектор на **каждый токен**:
`last_hidden_state` формы `[батч, токены, 312]`.

In [9]:
tokenizer = AutoTokenizer.from_pretrained("cointegrated/rubert-tiny2")
encoder = AutoModel.from_pretrained("cointegrated/rubert-tiny2").to(device).eval()


@torch.no_grad()
def token_states(texts):
    batch = tokenizer(texts, padding=True, return_tensors="pt").to(device)
    return batch, encoder(**batch).last_hidden_state


batch, states = token_states(["Отличное платье!"])
print(states.shape)

torch.Size([1, 6, 312])


### Одно слово — разные векторы

В word2vec у слова «замок» один вектор. У BERT вектор токена зависит от соседей. Сравним «замок»
в трёх предложениях: два про дверь, одно про крепость.

In [10]:
sentences = ["Он закрыл замок на ключ.", "Сломался замок на входной двери.", "Старинный замок стоит на холме."]
batch, states = token_states(sentences)


def word_vector(i, word):
    """Средний вектор токенов, из которых состоит слово, в предложении i.

    offset_mapping — для каждого токена его начало и конец в исходной строке (у [CLS] и [SEP] — (0, 0)).
    """
    offsets = tokenizer(sentences[i], return_offsets_mapping=True)["offset_mapping"]
    start = sentences[i].index(word)
    end = start + len(word)
    positions = [t for t, (a, b) in enumerate(offsets) if b > a and a >= start and b <= end]
    return states[i, positions].mean(dim=0)


v = [word_vector(i, "замок") for i in range(3)]
print(f"замок (дверь) — замок (дверь):    {F.cosine_similarity(v[0], v[1], dim=0):.3f}")
print(f"замок (дверь) — замок (крепость): {F.cosine_similarity(v[0], v[2], dim=0):.3f}")

замок (дверь) — замок (дверь):    0.889
замок (дверь) — замок (крепость): 0.781


Векторы одного и того же слова в двух «дверных» предложениях ближе, чем в «дверном» и «крепостном».
Эмбеддинг зависит от контекста — главное отличие трансформеров от word2vec.

### Эмбеддинг всего текста

Чтобы получить один вектор на текст, векторы токенов усредняют с учётом маски (mean pooling).
У rubert-tiny2 авторы рекомендуют вектор токена `[CLS]` — сравним оба.

In [11]:
def embed_texts(texts, pooling="mean"):
    batch, states = token_states(texts)
    if pooling == "cls":
        vec = states[:, 0]
    else:
        mask = batch["attention_mask"].unsqueeze(-1).float()
        vec = (states * mask).sum(dim=1) / mask.sum(dim=1)
    return F.normalize(vec, dim=1).cpu()


phrases = ["Платье маломерит, берите на размер больше", "Размер меньше заявленного",
           "Доставили очень быстро", "Посылка пришла за неделю", "Ткань тонкая и просвечивает"]
for pooling in ["mean", "cls"]:
    E = embed_texts(phrases, pooling)
    print(f"\n{pooling}-pooling, косинусы:")
    print(pd.DataFrame((E @ E.T).numpy(), index=[p[:22] for p in phrases], columns=range(5)).round(2))


mean-pooling, косинусы:
                           0     1     2     3     4
Платье маломерит, бери  1.00  0.60  0.37  0.43  0.44
Размер меньше заявленн  0.60  1.00  0.35  0.38  0.43
Доставили очень быстро  0.37  0.35  1.00  0.49  0.41
Посылка пришла за неде  0.43  0.38  0.49  1.00  0.39
Ткань тонкая и просвеч  0.44  0.43  0.41  0.39  1.00

cls-pooling, косинусы:
                           0     1     2     3     4
Платье маломерит, бери  1.00  0.76  0.57  0.53  0.59
Размер меньше заявленн  0.76  1.00  0.56  0.50  0.54
Доставили очень быстро  0.57  0.56  1.00  0.57  0.61
Посылка пришла за неде  0.53  0.50  0.57  1.00  0.48
Ткань тонкая и просвеч  0.59  0.54  0.61  0.48  1.00


Фразы про размер ближе друг к другу, чем к фразам про доставку. Модель всего на 29 млн параметров уже
даёт полезные эмбеддинги. В M11 мы сравним её с bge-m3 на задаче поиска.

## 4. datasets

Библиотека `datasets` хранит данные в формате Apache Arrow. Файлы отображаются в память, а не
загружаются целиком, поэтому работают и датасеты больше оперативной памяти.

In [12]:
ds = load_dataset("json", data_files={s: str(folder / f"{s}.jsonl") for s in ("train", "validation", "test")})
print(ds)
print(ds["train"].features)

DatasetDict({
    train: Dataset({
        features: ['text', 'label_text', 'label', 'id'],
        num_rows: 45000
    })
    validation: Dataset({
        features: ['text', 'label_text', 'label', 'id'],
        num_rows: 15000
    })
    test: Dataset({
        features: ['text', 'label_text', 'label', 'id'],
        num_rows: 15000
    })
})
{'text': Value('string'), 'label_text': Value('string'), 'label': Value('int64'), 'id': Value('string')}


`map` применяет функцию к датасету батчами и кэширует результат на диск: повторный запуск берёт готовое.

In [13]:
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)


tokenized = ds.map(tokenize, batched=True, remove_columns=["text", "label_text", "id"])
print(tokenized["train"])
lengths = np.array([len(x) for x in tokenized["train"]["input_ids"]])
print(f"токенов в отзыве: медиана {np.median(lengths):.0f}, обрезано до 128: {(lengths == 128).mean():.1%}")

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 45000
})


токенов в отзыве: медиана 24, обрезано до 128: 2.0%


После токенизации в датасете остались числа: `input_ids`, `attention_mask` и метка. Ровно в таком виде
данные пойдут в дообучение в уроке 3. Длину 128 выбрали по распределению: обрезаются 2% самых длинных
отзывов.

## 5. Hub: что проверить до использования модели

In [14]:
rows = []
for name in ["cointegrated/rubert-tiny2", "seara/rubert-tiny2-russian-sentiment", "openai-community/gpt2",
             "Qwen/Qwen3-8B", "BAAI/bge-m3"]:
    info = model_info(name, files_metadata=True)
    files = {s.rfilename: s.size or 0 for s in info.siblings}
    weights = [n for n in files if n.endswith(".safetensors")] or [n for n in files if n.endswith(".bin")]
    size = sum(files[n] for n in weights)
    rows.append({"модель": name, "лицензия": info.card_data.license if info.card_data else None,
                 "задача": info.pipeline_tag, "веса, ГБ": round(size / 1e9, 2), "ревизия": info.sha[:10]})
pd.DataFrame(rows).set_index("модель")

,лицензия,задача,"веса, ГБ",ревизия
модель,,,,
cointegrated/rubert-tiny2,mit,sentence-similarity,0.12,e8ed3b0c8b
seara/rubert-tiny2-russian-sentiment,mit,text-classification,0.12,ad9be38671
openai-community/gpt2,mit,text-generation,0.55,607a30d783
Qwen/Qwen3-8B,apache-2.0,text-generation,16.38,b968826d9c
BAAI/bge-m3,mit,sentence-similarity,2.27,5617a9f61b


Для воспроизводимости модель загружают с фиксированной ревизией: `AutoModel.from_pretrained(name,
revision="…")`. Иначе автор обновит веса, и результаты незаметно изменятся. Колонка «лицензия» — первое,
на что смотрят перед коммерческим использованием: у всех моделей в таблице она разрешающая
(MIT, Apache 2.0).

## ✅ Итоги

- Токенизатор определяет цену текста: русский в GPT-2 в разы дороже, чем в Qwen3.
- Чат-шаблон — это то, во что превращаются сообщения API перед подачей в модель.
- `pipeline` — быстро, но метки сопоставляют по именам, а качеству готовых моделей на своём тесте
  доверяют только после проверки данных обучения.
- `AutoModel` даёт контекстные векторы токенов, из них — эмбеддинги текстов.
- `datasets` + `map` — стандартный способ готовить данные для дообучения.

✍️ **Упражнение:** `exercises/ex02_bpe.py` — обучить BPE-токенизатор с нуля.